<a href="https://colab.research.google.com/github/dheerajnalla09/AIML/blob/main/fake_news_prediction_lstm_97_accurate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as ny

In [2]:
import pandas as pd

In [3]:
from google.colab import files
uploaded = files.upload()

Saving News _dataset.zip to News _dataset.zip


In [4]:
!unzip "News _dataset.zip" -d data

Archive:  News _dataset.zip
   creating: data/News _dataset/
   creating: data/News _dataset/Fake.csv/
  inflating: data/News _dataset/Fake.csv/Fake.csv  
   creating: data/News _dataset/True.csv/
  inflating: data/News _dataset/True.csv/True.csv  


In [6]:
!ls data
!ls "data/News _dataset"

'News _dataset'
Fake.csv  True.csv


In [9]:
import pandas as pd

fake = pd.read_csv("data/News _dataset/Fake.csv/Fake.csv")
true = pd.read_csv("data/News _dataset/True.csv/True.csv")

In [10]:
fake.head()
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [11]:
!mv "data/News _dataset/Fake.csv/Fake.csv" data/Fake.csv
!mv "data/News _dataset/True.csv/True.csv" data/True.csv

Move files out of folders:


In [12]:
fake = pd.read_csv("data/Fake.csv")
true = pd.read_csv("data/True.csv")

problem defination: Fake news refers to deliberately fabricated or misleading information presented as factual news. It often aims to deceive readers or viewers for various purposes, such as influencing opinions, spreading propaganda, or generating revenue through clicks. Detecting fake news is crucial in combating misinformation and maintaining the integrity of journalism and public discourse. Deep learning techniques, such as natural language processing and neural networks, can be effective in identifying patterns and features indicative of fake news, helping to automate the process of verification and fact-checking.

What you will find in this notebook📒-
Utilizing LSTM (Long Short-Term Memory) RNN for Fake News Detection.
Feature embedding is constructed before LSTM, facilitating data representation.
Preprocessing includes stemming SnowballStemmer regex using stemming SnowballStemmer regex and stemming SnowballStemmer regex cleaning.
One-hot encoding by Keras prepares textual data for LSTM input.
Adam optimizer and binary_crossentropy loss function are employed during model compilation.
Model performance evaluation includes classification report and confusion matrix.
Determination of the optimal threshold value for prediction refinement.

In [14]:
# importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from keras import Sequential
from keras.layers import Embedding, Dense, LSTM
from keras.utils import pad_sequences
import nltk
from nltk.stem.snowball import SnowballStemmer
import regex as re
from nltk.tokenize import sent_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
from nltk.corpus import stopwords

In [16]:
# download some packages
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

stop_words = stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [20]:
import pandas as pd

df_fake = pd.read_csv("data/Fake.csv")
df_true = pd.read_csv("data/True.csv")

In [22]:
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [23]:
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [24]:
# label them seperately
df_true['status'] = 0
df_fake['status'] = 1

In [25]:
# merge and remove unnecessary columns
df = pd.concat([df_true,df_fake])
df.drop(['subject','text','date'],axis=1,inplace=True)

Since we are going to build model only based on the title feature, hence dropped text, date , subject

In [26]:
# let's blend the smoothie
random_indexes = np.random.randint(0,len(df),len(df))
df = df.iloc[random_indexes].reset_index(drop=True)

Text analysis

In [27]:
pd.set_option('display.max_colwidth', 500)
random = np.random.randint(0,len(df),20)
df.iloc[random]

,title,status
16846,Britain preparing to transfer 400 million pounds to Iran - Telegraph newspaper,0
25233,Bullets and burns: Portraits of injured Rohingya refugees,0
14642,[VIDEO] DINESH D’SOUZA Warned Us About What The World Would Look Like If We Gave Obama Another Term In “2016: Obama’s America”…Was He Correct?,1
10650,Sarah Palin Silent As Texas Republicans Strip Medicaid Funding From Children With Disabilities,1
14052,"In record year for political ads, media buyers see tight market",0
31353,"REMEMBER WHEN Democrats, Media Mocked Betsy DeVos For Suggesting Schools May Need Guns To Protect Against Bears?…Look What Just Happened!",1
44650,Here’s What The World’s Top Economist Is Saying About Bernie Sanders,1
39500,"India's Modi remains overwhelmingly popular, says Pew poll",0
9948,WATCH: 75-YR OLD TRUMP SUPPORTER Says She’d Rather Go To Jail Than Remove Trump Signs From Front Yard,1
23861,Trump and May to discuss expanding U.S.-British trade: White House,0


work needs to be done on data before feeding to neural network-
Remove punctuations eg ""
Convert uppercase to lowercase
No need to apply stemming, otherwise it will just shorten the word unnecessary
Apply lemmatization
Remove all the stopwords
Finally make vocabulary after completion of 5 steps

In [28]:
# Null values
df.isnull().sum()

,0
title,0
status,0


In [29]:
# longest sentence length
def longest_sentence_length(text):
  return len(text.split())

df['maximum_length'] = df['title'].apply(lambda x : longest_sentence_length(x))
print('longest sentence having length -')
max_length = max(df['maximum_length'].values)
print(max_length)

longest sentence having length -
42


In [30]:
# Text cleaning
text_cleaning = "\b0\S*|\b[^A-Za-z0-9]+"

def preprocess_filter(text, stem=False):
  text = re.sub(text_cleaning, " ",str(text.lower()).strip())
  tokens = []
  for token in text.split():
    if token not in stop_words:
      if stem:
        stemmer = SnowballStemmer(language='english')
        token = stemmer.stem(token)
      tokens.append(token)
  return " ".join(tokens)

In [31]:
# Word embedding with pre padding
def one_hot_encoded(text,vocab_size=5000,max_length = 40):
    hot_encoded = one_hot(text,vocab_size)
    return hot_encoded

In [32]:
# word embedding pipeline
def word_embedding(text):
    preprocessed_text=preprocess_filter(text)
    return one_hot_encoded(preprocessed_text)

In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=40, input_length=max_length))
model.add(LSTM(100))
model.add(Dense(1, activation='sigmoid'))

model.build(input_shape=(None, max_length))   # 👈 important

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 42, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100)            │        56,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,501 (1001.96 KB)

 Trainable params: 256,501 (1001.96 KB)

 Non-trainable params: 0 (0.00 B)

The model utilizes a vocabulary size of 5000, reflecting the extensive nature of the dataset and the need to handle a wide range of words effectively.
By embedding input tokens into 40-dimensional vectors, the model captures nuanced semantic relationships, crucial for understanding the complex language patterns present in the dataset.
Leveraging an LSTM layer with 100 units, the model effectively learns from the extensive sequential data, ensuring it captures long-term dependencies and context effectively.
With a final dense layer employing a sigmoid activation function, the model delivers binary classification predictions, adeptly classifying the vast and varied dataset with accuracy.


In [39]:
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [41]:
# One hot encoded title
one_hot_encoded_title =df['title'].apply(lambda x : word_embedding(x)).values

In [42]:
# padding to make the size equal of the sequences
padded_encoded_title = pad_sequences(one_hot_encoded_title,maxlen=max_length,padding = 'pre')

In [43]:
# Splitting
X = padded_encoded_title
y = df['status'].values
y = np.array(y)

# shapes
print(X.shape)
print(y.shape)

(44898, 42)
(44898,)


In [44]:
# shape and size
print('X shape {}'.format(X.shape))
print('y shape {}'.format(y.shape))

X shape (44898, 42)
y shape (44898,)


In [45]:
# Splitting into training, testing
X_train,X_test,y_train,y_test = train_test_split(X,y, random_state = 42)

# Shape and size of train and test dataset
print('X train shape {}'.format(X_train.shape))
print('X test shape {}'.format(X_test.shape))
print('y train shape {}'.format(y_train.shape))
print('y test shape {}'.format(y_test.shape))

X train shape (33673, 42)
X test shape (11225, 42)
y train shape (33673,)
y test shape (11225,)


In [46]:
# Model training
# training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=15,batch_size=64)

Epoch 1/15
527/527 ━━━━━━━━━━━━━━━━━━━━ 38s 66ms/step - accuracy: 0.9173 - loss: 0.1985 - val_accuracy: 0.9522 - val_loss: 0.1256
Epoch 2/15
527/527 ━━━━━━━━━━━━━━━━━━━━ 39s 74ms/step - accuracy: 0.9724 - loss: 0.0769 - val_accuracy: 0.9656 - val_loss: 0.0976
Epoch 3/15
527/527 ━━━━━━━━━━━━━━━━━━━━ 41s 77ms/step - accuracy: 0.9857 - loss: 0.0422 - val_accuracy: 0.9693 - val_loss: 0.1102
Epoch 4/15
527/527 ━━━━━━━━━━━━━━━━━━━━ 36s 68ms/step - accuracy: 0.9914 - loss: 0.0272 - val_accuracy: 0.9700 - val_loss: 0.1251
Epoch 5/15
527/527 ━━━━━━━━━━━━━━━━━━━━ 35s 67ms/step - accuracy: 0.9948 - loss: 0.0161 - val_accuracy: 0.9707 - val_loss: 0.1326
Epoch 6/15
527/527 ━━━━━━━━━━━━━━━━━━━━ 36s 67ms/step - accuracy: 0.9962 - loss: 0.0132 - val_accuracy: 0.9694 - val_loss: 0.1462
Epoch 7/15
527/527 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.9977 - loss: 0.0075 - val_accuracy: 0.9668 - val_loss: 0.1772
Epoch 8/15
527/527 ━━━━━━━━━━━━━━━━━━━━ 39s 74ms/step - accuracy: 0.9980 - loss: 0.0066 - 

Evaluation

In [48]:
# setting threshold value
def best_threshold_value(thresholds:list,X_test):
    accuracies = []
    for thresh in thresholds:
        ypred =model.predict(X_test)
        ypred = np.where(ypred> thresh,1,0)
        accuracies.append(accuracy_score(y_test,ypred))
    return pd.DataFrame({
        'Threshold': thresholds,
        'Accuracy' : accuracies
    })

In [49]:
best_threshold_value([0.4,0.5,0.6,0.7,0.8,0.9], X_test)

351/351 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step
351/351 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step
351/351 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step
351/351 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step
351/351 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step
351/351 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step


,Threshold,Accuracy
0,0.4,0.972027
1,0.5,0.971759
2,0.6,0.971938
3,0.7,0.971492
4,0.8,0.971759
5,0.9,0.971225


Not much difference in accuray
But the most suitable threshold value we have got is 0.4

In [50]:
# Predictino value at threshold 0.4
y_pred = model.predict(X_test)
y_pred = np.where(y_pred >0.4, 1, 0)

351/351 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step


In [51]:
# Confusion matrix
print('Confusion matrix')
print(confusion_matrix(y_pred,y_test))
print('----------------')
print('Classification report')
print(classification_report(y_pred,y_test))

Confusion matrix
[[5142  154]
 [ 160 5769]]
----------------
Classification report
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      5296
           1       0.97      0.97      0.97      5929

    accuracy                           0.97     11225
   macro avg       0.97      0.97      0.97     11225
weighted avg       0.97      0.97      0.97     11225




The model performs well in both classes, with high precision, recall, and F1-score, suggesting robustness in classification.
There is no significant imbalance in performance between the two classes, as evidenced by similar metrics for both classes.
The model's overall performance is excellent, achieving high accuracy on the dataset

**Predictions**

In [52]:
# input generator
def prediction_input_processing(text):
    encoded = word_embedding(text)
    padded_encoded_title = pad_sequences([encoded],maxlen=max_length,padding = 'pre')
    output = model.predict(padded_encoded_title)
    output = np.where(0.4>output,1,0)
    if output[0][0] == 1:
        return 'Yes this News is fake'
    return 'No, It is not fake'

In [53]:
# predictions
prediction_input_processing('Americans are more concerned over Indians fake open source contribution')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step


'Yes this News is fake'

In [54]:
news = 'Trump Just Sent Michelle Obama a Bill She will Never Be able to pay in her lifetime'
prediction_input_processing(news)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step


'No, It is not fake'